<a href="https://colab.research.google.com/github/ddy623/Kaggle-Projects/blob/main/avocado_sales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Read Me: Avocado Price Prediction Machine Learning Project

This notebook performs a machine learning task to predict avocado prices using a dataset loaded from Google Drive.

## Project Steps:

1.  **Data Loading**: The `avocado.csv` dataset was loaded from Google Drive into a pandas DataFrame.
2.  **Initial Data Inspection**: The `df.info()`, `df.describe()`, `df.duplicated().sum()`, and `df.isna().sum()` methods were used to understand the data's structure, identify missing values, and check for duplicates.
3.  **Data Preprocessing and Feature Engineering**:
    *   Irrelevant columns (`'Unnamed: 0'`, `'Total Volume'`, `'Total Bags'`) were dropped.
    *   The `'Date'` column was converted to datetime objects.
    *   New numerical features, `'Month'` and `'Day'`, were extracted from the `'Date'` column.
    *   The original `'Date'` column was then dropped.
4.  **Target and Feature Definition**:
    *   The target variable `y` was set to `'AveragePrice'`.
    *   The feature set `X` included all other relevant columns.
5.  **Machine Learning Pipeline**: A scikit-learn pipeline was constructed to streamline the machine learning workflow:
    *   **Preprocessing**: `StandardScaler` was applied to numerical features, and `OneHotEncoder` was used for categorical features (`'type'` and `'region'`).
    *   **Model**: A `RandomForestRegressor` was chosen as the prediction model.
6.  **Model Training and Evaluation**:
    *   The data was split into training and testing sets (80% train, 20% test) using `train_test_split`.
    *   The `RandomForestRegressor` model was trained on the training data.
    *   Predictions were made on the test set.
    *   The model's performance was evaluated using:
        *   **Mean Absolute Error (MAE)**: Measures the average magnitude of the errors in a set of predictions, without considering their direction.
        *   **R-squared (R2)**: Represents the proportion of the variance in the dependent variable that is predictable from the independent variables.

## Results:

The trained `RandomForestRegressor` model achieved the following performance metrics:

*   **Mean Absolute Error (MAE): 0.11**
*   **R-squared (R2): 0.85**

These results indicate that the model's predictions for avocado prices are, on average, off by approximately $0.11, and it explains about 85% of the variance in avocado prices within this dataset. This suggests a reasonably good predictive performance.

In [24]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:


# IMPORTANT: Replace 'path/to/your/file.csv' with the actual path to your CSV file on Google Drive
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Data Science Project/avocado.csv')

# Display the first 5 rows of the DataFrame
display(df.head())

,Unnamed: 0,Date,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,type,year,region
0,0,2015-12-27,1.33,64236.62,1036.74,54454.85,48.16,8696.87,8603.62,93.25,0.0,conventional,2015,Albany
1,1,2015-12-20,1.35,54876.98,674.28,44638.81,58.33,9505.56,9408.07,97.49,0.0,conventional,2015,Albany
2,2,2015-12-13,0.93,118220.22,794.70,109149.67,130.50,8145.35,8042.21,103.14,0.0,conventional,2015,Albany
3,3,2015-12-06,1.08,78992.15,1132.00,71976.41,72.58,5811.16,5677.40,133.76,0.0,conventional,2015,Albany
4,4,2015-11-29,1.28,51039.60,941.48,43838.39,75.78,6183.95,5986.26,197.69,0.0,conventional,2015,Albany


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18249 entries, 0 to 18248
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Unnamed: 0    18249 non-null  int64  
 1   Date          18249 non-null  object 
 2   AveragePrice  18249 non-null  float64
 3   Total Volume  18249 non-null  float64
 4   4046          18249 non-null  float64
 5   4225          18249 non-null  float64
 6   4770          18249 non-null  float64
 7   Total Bags    18249 non-null  float64
 8   Small Bags    18249 non-null  float64
 9   Large Bags    18249 non-null  float64
 10  XLarge Bags   18249 non-null  float64
 11  type          18249 non-null  object 
 12  year          18249 non-null  int64  
 13  region        18249 non-null  object 
dtypes: float64(9), int64(2), object(3)
memory usage: 1.9+ MB


In [17]:
df.describe()

,Unnamed: 0,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,year
count,18249.000000,18249.000000,1.824900e+04,1.824900e+04,1.824900e+04,1.824900e+04,1.824900e+04,1.824900e+04,1.824900e+04,18249.000000,18249.000000
mean,24.232232,1.405978,8.506440e+05,2.930084e+05,2.951546e+05,2.283974e+04,2.396392e+05,1.821947e+05,5.433809e+04,3106.426507,2016.147899
std,15.481045,0.402677,3.453545e+06,1.264989e+06,1.204120e+06,1.074641e+05,9.862424e+05,7.461785e+05,2.439660e+05,17692.894652,0.939938
min,0.000000,0.440000,8.456000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,2015.000000
25%,10.000000,1.100000,1.083858e+04,8.540700e+02,3.008780e+03,0.000000e+00,5.088640e+03,2.849420e+03,1.274700e+02,0.000000,2015.000000
50%,24.000000,1.370000,1.073768e+05,8.645300e+03,2.906102e+04,1.849900e+02,3.974383e+04,2.636282e+04,2.647710e+03,0.000000,2016.000000
75%,38.000000,1.660000,4.329623e+05,1.110202e+05,1.502069e+05,6.243420e+03,1.107834e+05,8.333767e+04,2.202925e+04,132.500000,2017.000000
max,52.000000,3.250000,6.250565e+07,2.274362e+07,2.047057e+07,2.546439e+06,1.937313e+07,1.338459e+07,5.719097e+06,551693.650000,2018.000000


In [18]:
df.duplicated().sum()

np.int64(0)

In [19]:
null_sums = df.isna().sum()
null_sums

,0
Unnamed: 0,0
Date,0
AveragePrice,0
Total Volume,0
4046,0
4225,0
4770,0
Total Bags,0
Small Bags,0
Large Bags,0


In [20]:
df = df.drop(columns=['Unnamed: 0', 'Total Volume', 'Total Bags'])

In [21]:
#confirm 'Unnamed: 0', Total Volume, and Total Bags was dropped
df.describe()

,AveragePrice,4046,4225,4770,Small Bags,Large Bags,XLarge Bags,year
count,18249.000000,1.824900e+04,1.824900e+04,1.824900e+04,1.824900e+04,1.824900e+04,18249.000000,18249.000000
mean,1.405978,2.930084e+05,2.951546e+05,2.283974e+04,1.821947e+05,5.433809e+04,3106.426507,2016.147899
std,0.402677,1.264989e+06,1.204120e+06,1.074641e+05,7.461785e+05,2.439660e+05,17692.894652,0.939938
min,0.440000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,2015.000000
25%,1.100000,8.540700e+02,3.008780e+03,0.000000e+00,2.849420e+03,1.274700e+02,0.000000,2015.000000
50%,1.370000,8.645300e+03,2.906102e+04,1.849900e+02,2.636282e+04,2.647710e+03,0.000000,2016.000000
75%,1.660000,1.110202e+05,1.502069e+05,6.243420e+03,8.333767e+04,2.202925e+04,132.500000,2017.000000
max,3.250000,2.274362e+07,2.047057e+07,2.546439e+06,1.338459e+07,5.719097e+06,551693.650000,2018.000000


Perform Machine Learning.
Define X and Y variables. X will be relevant columns and Y is target variable.

In [23]:
# Reload the dataframe to ensure 'Date' column is present and in a clean state
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Data Science Project/avocado.csv')

# Drop columns identified earlier
df = df.drop(columns=['Unnamed: 0', 'Total Volume', 'Total Bags'])

# Convert 'Date' to datetime objects and extract features
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df = df.drop(columns=['Date'])

# Define target variable (y) and features (X)
y = df['AveragePrice']
X = df.drop(columns=['AveragePrice'])

# Identify categorical and numerical features
categorical_features = ['type', 'region']
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
# Month and Day are numerical features and should be included by select_dtypes now.

# Create a preprocessor for one-hot encoding categorical features and scaling numerical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Create a machine learning pipeline
model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('regressor', RandomForestRegressor(random_state=42))])

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
model_pipeline.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model_pipeline.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"R-squared (R2): {r2:.2f}")

Mean Absolute Error (MAE): 0.11
R-squared (R2): 0.85
